In [ ]:
!pip install sentence-transformers pandas pyarrow scikit-learn emoji tqdm

import os
import pandas as pd
import numpy as np
import emoji
from google.colab import drive
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.0 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
model = SentenceTransformer('sentence-transformers/all-distilroberta-v1', device='cuda')

base_path = "/content/drive/MyDrive/Reddit_Project/embedding_data"
subreddits_folders = [f for f in os.listdir(base_path) if f.startswith("subreddit=")]

print(f"Tập hợp {len(subreddits_folders)} Subreddits")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Tập hợp 3610 Subreddits


In [ ]:
subreddit_vectors = []

for sub_folder in tqdm(subreddits_folders, desc="Processing"):
    sub_name = sub_folder.split("=")[1]
    folder_path = os.path.join(base_path, sub_folder)

    try:
        df_sub = pd.read_parquet(folder_path)
    except Exception:
        continue

    if df_sub.empty or 'title' not in df_sub.columns:
        continue

    df_sub['title'] = df_sub['title'].astype(str).apply(lambda x: emoji.demojize(x, delimiters=(" ", " ")))
    titles = df_sub['title'].dropna().tolist()

    if not titles:
        continue

    embeddings = model.encode(titles, batch_size=128, show_progress_bar=False)
    mean_vec = np.mean(embeddings, axis=0)

    subreddit_vectors.append({
        'subreddit': sub_name,
        'mean_embedding': mean_vec
    })

df_vectors = pd.DataFrame(subreddit_vectors)

print("\nHoàn thành embedding")

Processing:   0%|          | 0/3610 [00:00<?, ?it/s]


Hoàn thành embedding


In [ ]:
print("Tính độ tương đồng Cosine")

emb_matrix = np.stack(df_vectors['mean_embedding'].values)
subreddits_list = df_vectors['subreddit'].tolist()

sim_matrix = cosine_similarity(emb_matrix)

results = []
num_subs = len(subreddits_list)

for i in range(num_subs):
    for j in range(i + 1, num_subs):
        score = sim_matrix[i][j]
        if score > 0.3:
            results.append({
                "Subreddit_A": subreddits_list[i],
                "Subreddit_B": subreddits_list[j],
                "Similarity_Score": round(float(score), 4)
            })

df_results = pd.DataFrame(results)
df_results_sorted = df_results.sort_values(by="Similarity_Score", ascending=False).reset_index(drop=True)

save_path = "/content/drive/MyDrive/Reddit_Project/subreddit_similarity_results.csv"
df_results_sorted.to_csv(save_path, index=False)

print(f"\File được lưu tại: {save_path}")

<>:27: SyntaxWarning: invalid escape sequence '\F'
<>:27: SyntaxWarning: invalid escape sequence '\F'
/tmp/ipykernel_546/570099191.py:27: SyntaxWarning: invalid escape sequence '\F'
  print(f"\File được lưu tại: {save_path}")


Tính độ tương đồng Cosine
\File được lưu tại: /content/drive/MyDrive/Reddit_Project/subreddit_similarity_results.csv
